# House Price Prediction: Complete End-to-End Analytics & Modeling
### **Project Objective:** A full-lifecycle analysis—from raw data exploration to high-performance model deployment.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import joblib
import os
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
print("Environment Ready.")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/house_data.csv')
display(df.head())
print(f"Data Shape: {df.shape}")

## 3. Basic Data Understanding

In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print("\n--- Statistics ---")
display(df.describe().T)

## 4. Data Quality Check

In [ ]:
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

## 5. Data Cleaning

In [ ]:
df = df.drop_duplicates()
df = df[df['area_sqft'] > 0]
print("Cleaning complete.")

## 6. Target Variable Analysis (Price)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['price'], kde=True, color='teal')
plt.title('House Price Distribution')
plt.show()

## 7. Univariate Analysis

In [ ]:
cols = ['bedrooms', 'bathrooms', 'location']
for col in cols:
    plt.figure(figsize=(8, 4))
    sns.countplot(x=col, data=df, palette='viridis')
    plt.show()

## 8. Bivariate Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='area_sqft', y='price', data=df, hue='location')
plt.title('Area vs Price')
plt.show()

## 9. Multivariate Analysis

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='magma')
plt.show()

## 10. Outlier Detection & Treatment

In [ ]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df['price'] < (Q1 - 1.5 * IQR)) | (df['price'] > (Q3 + 1.5 * IQR)))]
print("Outliers treated using IQR.")

## 11. Feature Engineering

In [ ]:
df['house_age'] = 2025 - df['year_built']
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['price_per_sqft'] = df['price'] / df['area_sqft']
print("Engineered Features: house_age, total_rooms, price_per_sqft")

## 12. Advanced Business Insights

In [ ]:
display(df.groupby('location')['price'].mean().sort_values(ascending=False))

## 13. Data Preparation for Modeling

In [ ]:
X = df.drop(['price', 'price_per_sqft'], axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ['location', 'furnishing', 'property_type']
num_cols = ['area_sqft', 'bedrooms', 'bathrooms', 'floors', 'parking', 'year_built', 'house_age', 'total_rooms']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## 14. Detailed Model Training Steps
We will now train 9 individual regression models as part of our comparative analysis.

### Step 14.1: Linear, Ridge, and Lasso Regression

In [ ]:
lr_p = Pipeline([('pre', preprocessor), ('reg', LinearRegression())]).fit(X_train, y_train)
ridge_p = Pipeline([('pre', preprocessor), ('reg', Ridge())]).fit(X_train, y_train)
lasso_p = Pipeline([('pre', preprocessor), ('reg', Lasso())]).fit(X_train, y_train)
print("Linear models trained.")

### Step 14.2: Decision Tree & Random Forest

In [ ]:
dt_p = Pipeline([('pre', preprocessor), ('reg', DecisionTreeRegressor())]).fit(X_train, y_train)
rf_p = Pipeline([('pre', preprocessor), ('reg', RandomForestRegressor())]).fit(X_train, y_train)
print("Tree models trained.")

### Step 14.3: Boosting Models (GB, Ada, ET, XGB)

In [ ]:
gb_p = Pipeline([('pre', preprocessor), ('reg', GradientBoostingRegressor())]).fit(X_train, y_train)
ada_p = Pipeline([('pre', preprocessor), ('reg', AdaBoostRegressor())]).fit(X_train, y_train)
et_p = Pipeline([('pre', preprocessor), ('reg', ExtraTreesRegressor())]).fit(X_train, y_train)
xgb_p = Pipeline([('pre', preprocessor), ('reg', XGBRegressor())]).fit(X_train, y_train)
print("Boosting models trained.")

## 15. Evaluation & Model Persistence

In [ ]:
final_model = Pipeline([('pre', preprocessor), ('reg', XGBRegressor())])
final_model.fit(X, y)
joblib.dump(final_model, '../model/house_model.pkl')
print("Model persistent at model/house_model.pkl")

## 16. Final Summary
- Dataset cleaned and EDA completed.
- 9 Regression models compared.
- High-performance XGBoost pipeline saved for deployment.